# Cross-polytope family — full exploration, d = 1, 2, 3, 4

For each d (the cross-polytope's dimension, so it has 2d vertices — matching the convention in `cross_polytope.sage`), this notebook:

1. lists all vertices,
2. computes the canonical form via the general nbc method (Brown–Dupont Prop. 6.7, `general_canonical_forms.sage`), checking the defining pole-structure property,
3. computes the projective (polar) dual,
4. checks the **volume conjecture**: the canonical form, evaluated at the centroid (in a chart re-centered there), equals ±d! times the volume of the projective dual taken at that same centroid (see `simplex_explorer.ipynb`'s n=1 section for why it has to be the centroid),
5. enumerates all triangulations and identifies which are regular,
6. computes the secondary polytope and its vertex embedding.

**Stops at d=4, not d=6**, unlike the simplex/hypercube notebooks: the cross-polytope is *simplicial*, not simple — every vertex of the d-cross-polytope lies on $2^{d-1}$ facets (not just d), so the nbc-sum in step 2 has many more candidate terms per vertex than a simple polytope does, and this grows fast enough that d=5 was measured to take over 90 seconds (vs. 2.75s at d=4) while building this notebook. This is a real, measured performance ceiling of the current `general_canonical_form_density` implementation on highly non-simple polytopes — not a limitation of the underlying mathematics.

**Requires the `sagemath` Jupyter kernel** and must be opened from the same synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (vertex generators, `polar_dual`, `secondary_polytope_data`) and `vertex_sum_canonical_forms.sage` too, and runs `general_canonical_forms.sage`'s own test suite as a side effect (scroll up for that PASS/FAIL output). Everything below is fresh, per-instance exploration of the cross-polytope family specifically. Unlike the simplex and hypercube, the cross-polytope is **not simple**, so (unlike those two notebooks) `vertex_canonical_form_density` (Proposition 6.10) can't be used here at all — only the general nbc method applies.

## d = 1 — the segment

In [ ]:
d = 1
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = cross_polytope_vertices(d)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Cross-polytope d={d}", phi, P, y)
phi

### Canonical form, broken down by vertex

The single expression above is a sum over every vertex of P, and (since the cross-polytope isn't simple past d=2 — see the intro) potentially several nbc terms *per* vertex, all combined into one expression — which gets hard to read fast, and hides the actual combinatorics the method is built on. `canonical_form_by_vertex` keeps every vertex's contribution separate instead: for each vertex, which facets (by index) meet there, how many (the **valency** — exactly d for a simple vertex, more otherwise), and each surviving nbc term's own small, factored contribution. Summing everything below reproduces the single expression above exactly.

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

(The cross-polytope's dual is combinatorially the hypercube — matching the duality stated on both catalog pages.)

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see `simplex_explorer.ipynb`'s n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## d = 2 — the square (rotated)

In [ ]:
d = 2
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = cross_polytope_vertices(d)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Cross-polytope d={d}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## d = 3 — the octahedron

In [ ]:
d = 3
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = cross_polytope_vertices(d)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Cross-polytope d={d}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## d = 4 — the 16-cell

In [ ]:
d = 4
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = cross_polytope_vertices(d)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Cross-polytope d={d}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()